In [25]:
# Load Customs 2015 dataset
import pandas as pd

df = pd.read_csv(
    r"C:\Users\Lance Santos\Downloads\Data Folder\bettergov.ph 2015.csv",
    encoding="cp1252"
)

print(df.shape)
print(df.columns.tolist())

C:\Users\Lance Santos\AppData\Local\Temp\ipykernel_29868\1211923022.py:4: DtypeWarning: Columns (4: entry, 25: prefcode, 28: subport, 29: port) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


(2236612, 30)
['uid', 'ty', 'tq', 'tm', 'entry', 'hscode', 'goodsdescription', 'p', 'q', 'm_fob', 'm_cif', 'fx_usd', 'dutiablevalueforeign', 'exchangerate', 'currency', 'dutiablevaluephp', 'dutypaid', 'exciseadvalorem', 'arrastre', 'wharfage', 'vatbase', 'vatpaid', 'othertax', 'finesandpenalties', 'dutiestaxes', 'prefcode', 'countryorigin_iso3', 'countryexport_iso3', 'subport', 'port']


In [26]:
print(df["dutiablevaluephp"].dtype)
print(df["dutiablevaluephp"].describe())
print(df["dutiablevaluephp"].isna().sum())

int64
count    2.236612e+06
mean     1.603885e+06
std      2.479364e+07
min      1.000000e+00
25%      1.901500e+04
50%      1.016945e+05
75%      5.849100e+05
max      8.367496e+09
Name: dutiablevaluephp, dtype: float64
0


In [27]:
#convert to a NumPy array
import numpy as np

values = df["dutiablevaluephp"].dropna().to_numpy()

print(type(values))
print(values.shape)

<class 'numpy.ndarray'>
(2236612,)


In [28]:
#boolean mask + aggregation
threshold = 1000000

mask = values > threshold

filtered_values = values[mask]

print("Number above threshold:", len(filtered_values))
print("Total:", np.sum(filtered_values))



Number above threshold: 411305
Total: 3288821943420


In [29]:
#fixed seed sample
rng = np.random.default_rng(42)

sample = rng.choice(values, size=10000, replace=False)
print(sample.shape)

(10000,)


In [ ]:
#vectorized calculation / calculates 10% of each value in the sample at once using numpy

vectorized_result = sample* 0.10 
print(vectorized_result[:5])

[10328.6 28277.1  5562.4  4929.4  3778. ]


In [31]:
#same calculation with a loop

loop_result = []
for value in sample:
    loop_result.append(value * 0.10)

loop_result = np.array(loop_result)

print(loop_result[:5])

[10328.6 28277.1  5562.4  4929.4  3778. ]


In [32]:
#verify equal results
print(np.allclose(vectorized_result, loop_result))

True


In [43]:
#benchmark 5 runs
import time

loop_times = []
vectorized_times = []

for _ in range(5):
    # Time loop calculation
    start = time.perf_counter()

    result = []
    for value in sample:
        result.append(value * 0.10)

    result = np.array(result)

    end = time.perf_counter()
    loop_times.append(end - start)

    # Time vectorized calculation
    start = time.perf_counter()

    result = sample * 0.10

    end = time.perf_counter()
    vectorized_times.append(end - start)

print("Loop times:", loop_times)
print("Vectorized times:", vectorized_times)

print("Loop median:", np.median(loop_times))
print("Vectorized median:", np.median(vectorized_times))

Loop times: [0.021829599994816817, 0.01922770000237506, 0.01902439999685157, 0.017177100002299994, 0.017794100000173785]
Vectorized times: [6.839999696239829e-05, 4.410000110510737e-05, 4.2499988921917975e-05, 2.3100001271814108e-05, 3.060000017285347e-05]
Loop median: 0.01902439999685157
Vectorized median: 4.2499988921917975e-05


In [44]:
from src.numpy_ops import (
    apply_mask,
    vectorized_calculation,
    loop_calculation,
    aggregate_values
)

In [35]:
filtered_values = apply_mask(values, 1_000_000)

print("Number above threshold:", len(filtered_values))
print("Total:", aggregate_values(filtered_values))

Number above threshold: 411305
Total: 3288821943420.0


In [36]:
vectorized_result = vectorized_calculation(sample)
loop_result = loop_calculation(sample)

print(np.allclose(vectorized_result, loop_result))

True


In [37]:
import pandas as pd

top10 = pd.read_csv("top10.csv")
pivot = pd.read_csv("pivot.csv")

print("TOP10 shape:", top10.shape)
print("TOP10 columns:", top10.columns.tolist())

print("\nPIVOT shape:", pivot.shape)
print("PIVOT columns:", pivot.columns.tolist())

TOP10 shape: (10, 5)
TOP10 columns: ['countryorigin_iso3', 'row_count', 'valid_count', 'sum', 'mean']

PIVOT shape: (198, 6)
PIVOT columns: ['countryorigin_iso3', '2015q1', '2015q2', '2015q3', '2015q4', 'Total']


In [45]:
from src.plots import make_bar_plot, make_heatmap

In [47]:
make_bar_plot(
    data=top10,
    x_column="countryorigin_iso3",
    y_column="sum",
    title="Top 10 Countries by Dutiable Value",
    x_label="Country of Origin (ISO3)",
    y_label="Dutiable Value (PHP)",
    output_path="bar.png",
)

This shows the top 10 countries by total dutiable value in the Customs 2015 Summary

In [49]:
heatmap_data = pivot.set_index("countryorigin_iso3")[
    ["2015q1", "2015q2", "2015q3", "2015q4"]
].drop(index="Total")

In [50]:
make_heatmap(
    data=heatmap_data,
    title="Dutiable Value by Country of Origin and Quarter",
    x_label="Quarter",
    y_label="Country of Origin (ISO3)",
    output_path="heatmap.png",
)

Heatmap shows the dutitable value for each country of origin across the four quarters of 2015